## 1. 키 설정

In [ ]:
import re, os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import HumanMessage

#os.environ['OPENAI_API_KEY'] = ""

## 2. 시스템프롬프트 설정

In [3]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    request_timeout=60,
    api_key=os.environ["OPENAI_API_KEY"]
)

system_prompt = """
You are an assistant specialized in restoring omitted components in Korean dialogue lines and merging consecutive utterances by the same speaker.

*Core Rules (Crucial)*
1. You will receive a dialogue as plain text. Output only the dialogue, with all minimally necessary restorations inside square brackets [ ].
2. Merge consecutive lines from the same speaker into a single line. Keep the original order of phrases while merging. Separate clauses with natural spacing and punctuation.
3. Correct obvious typos, spacing, and malformed colloquialisms when it does not change meaning (e.g., “해소”→“해서”, “가티”→“같이”).
4. Do not over-restore. Preserve only essential sentence elements (subjects, objects, particles, predicates, key adjuncts). Do not resurrect trivial or redundant omissions (fillers, repeated function words, obvious ellipses) if they are unnecessary for clarity.
5. Do not change meaning, tense, or referents (“that/then/there”). If uncertain, prefer minimal restoration.
6. If multiple insertions are needed in a line, place each restored piece exactly where it belongs, each inside its own [ ].

*Processing Guidelines (for internal reasoning only)*
1. Use discourse context (topic continuity, before/after turns) to infer omitted elements, then restore only what is necessary for grammaticality and clarity using [ ].
2. Normalize colloquial contractions if meaning is unchanged (e.g., “했지” can remain as-is; add particles like “[가]”, “[을]” only when needed).
3. When merging same-speaker lines, keep laughter/interjections/emojis inline if they contribute to tone, but avoid duplicating them unnecessarily.

*Output Format*
1. Output only the dialogue, preserving speaker labels (화자 1:, 화자 2: …).
2. After merging, each speaker block should be a single line per contiguous turn (i.e., no duplicate consecutive lines for the same speaker).
3. All restorations must be shown inside square brackets [ ].
4. Do not include any explanations, headers, or metadata—only the merged, restored dialogue.

---

**Few-shot examples (mimic exactly this format)**

Example 1:
Input:
화자 2: 진짜 신의 한수
화자 1: 이사하자마자 비 많이 와서 베란다 물 많이 새는 거 알았잖아
화자 2: 글치 계속 해떴으면 몰랐겠지
화자 1: 그 때 물새는 거 알고 코킹작업해소 다행이다
화자 2: ㅇㅇ 안그랬으면 오늘처럼 비 많이 내리는 날 물바다됐을거야
화자 1: 요 아래 씽크홀 공사하던데 괜찮을라나
화자 2: 그러게 저번에도 비 많이 와서 땅꺼진 건데 큰일이네
화자 1: 하수도 공사도 같이 하더만 물 안빠져서
화자 2: 새로 지은 곳인데도 그러네
화자 1: 부실공사지 뭐
화자 2: 비 많이 올 때는 그쪽으로 다니지 말아야겠다
화자 1: ㅇㅇ 조심해
화자 1: 저번에 지나가다 보니 좀 무섭더라
화자 2: 나도 봤는데 씽크홀 크기가 엄청나더라
화자 1: 오늘 비가 엄청 많이 내리네

Output:
화자 2: 진짜 [이사한 게] 신의 한수[였어]
화자 1: [우리가] 이사하자마자 비[가] 많이 와서 베란다[에] 물[이] 많이 새는 거 알았잖아
화자 2: 글치 [그걸] 계속 [해가] 떴으면 [우리는] 몰랐겠지
화자 1: 그 때 [베란다에] 물 새는 거 알고 코킹 작업해[서] 다행이다
화자 2: ㅇㅇ 안 그랬으면 오늘처럼 비[가] 많이 내리는 날 [집이] 물바다 됐을 거야
화자 1: 요 아래 씽크홀 공사하던데 [그게] 괜찮을라나
화자 2: 그러게 저번에도 비[가] 많이 와서 땅[이] 꺼진 건데 큰일이네
화자 1: 하수도 공사도 같이 하더만 물[이] 안 빠져서
화자 2: [거기가] 새로 지은 곳인데도 그러네
화자 1: [그건] 부실 공사[지] 뭐
화자 2: 비[가] 많이 올 때는 [우리는] 그쪽으로 다니지 말아야겠다
화자 1: ㅇㅇ 조심해 저번에 [거길] 지나가다 보니 좀 무섭더라
화자 2: 나도 봤는데 [그] 씽크홀[의] 크기[가] 엄청나더라
화자 1: 오늘 비[가] 엄청 많이 내리네

Example 2
Input: 
화자 2: 어제다친발목에
화자 2: 파스를계속붙엿더니
화자 2: 따갑습니다요..
화자 2: ㄱㅋㄱㅋㄱㅋ
화자 1: ㅋㅋㅋ아
화자 1: 친구들이 아이폰인게 계속 슬프네요
화자 1: ㅋㅋㅋname2님이랑 가티 하차해야것어유
화자 1: ㅋㅋㅋㅋ
화자 1: 파스말구
화자 1: 온찜질!!
화자 2: 아 온찜질인가여..!!!!!!
화자 1: 수건물에적셔서 전자렌지 돌리세욥!!!!!
화자 2: 악저랑하차하신다니익!!!!!!
화자 2: 오 팁감사합니다 진짜하구자야겟어요
화자 2: ㅠㅡㅠ
화자 1: ㅎㅎ강추!!!!
화자 2: 나이들수록
화자 2: 다친곳 또 다치구
화자 2: 이런게많아지는거같아요
화자 1: 특히 발목은
화자 1: 더그래요ㅜㅜ
화자 2: 안그랬는데갑자기몇달전에
화자 2: 스피드민턴친다고설치다가 한번다치고나서
화자 2: 그때부터ㅋㅋㄲㅋ
화자 1: 아이공
화자 1: 온찜질 계속 해주세유
화자 2: 네 ㅠㅡㅠ  아직도 냉찜질온찜질 헷갈립니다..ㅋㅋㅋㅋ어디에뭐가좋은지
화자 1: ㅋㅋ발목을 애껴주세요
화자 1: ㅋㅋㅋㅋ
화자 2: 그러게요소중히다뤄야하는데ㅜㅜ

Output:
화자 2: 어제 [다친] 발목[이] 파스를 계속 붙였더니 따갑습니다[요].. ㄱㅋㄱㅋㄱㅋ
화자 1: ㅋㅋㅋ 아 친구들이 아이폰[을] [쓰고] 있는 게 계속 슬프네요 ㅋㅋㅋ name2님이랑 [같이] 하차해야겠어유 ㅋㅋㅋㅋ 파스 말고 온찜질!!
화자 2: 아, 온찜질인가여..!!!!!!
화자 1: 수건[을] 물에 적셔서 전자렌지 돌리세요!!!!!
화자 2: 악, 저랑 하차하신다니!!!!!! 오, 팁 감사합니다. 진짜 하고 자야겟어요 ㅠㅡㅠ
화자 1: ㅎㅎ 강추!!!!
화자 2: 나이 들수록 다친 곳 또 다치고 이런 게 많아지는 거 같아요
화자 1: 특히 발목은 더 그래요ㅜㅜ
화자 2: 안 그랬는데 갑자기 몇 달 전에 스피드민턴[을] 친다고 설치다가 한번 다치고 나서 그때부터 [다친 곳을 또 다치네요]  ㅋㅋㄲㅋ
화자 1: 아이공 온찜질 [발목에] 계속 해주세유
화자 2: 네 ㅠㅡㅠ  아직도 냉찜질온찜질 [어디에 어떤게 좋은지] 헷갈립니다..ㅋㅋㅋㅋ어디에 [냉온 찜질중에] 뭐가좋은지
화자 1: ㅋㅋ발목을 애껴주세요 ㅋㅋㅋㅋ
화자 2: 그러게요 [발목을] 소중히다뤄야하는데ㅜㅜ
"""

integrated_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt.strip()),
    ("human", "<Actual Dialouge to Process>\n{actual_dialouge}")
])

chain = integrated_prompt | llm

## 3. 실제 대화 데이터셋 수정해보기

### 3.1 nikluge-2025-대화 맥락 추론-train-000342

In [ ]:
dialouge1 = '''
화자 1: 요즘 따라 여행가고 싶네요 ㅜㅜ
화자 2: 저도요ㅠㅠ
화자 2: 못나가서 더 그런걸까요?
화자 1: 맞아요 ㅜㅜ
화자 1: 코로나 지금 거의 2년 다되가잖아요
화자 1: 다들 여행이 근질근질 할거에요
화자 2: 맞아요ㅠ
화자 2: 해외는 진짜 꿈이에요
화자 1: 해외보단 국내가 그래도 좀 안전한 거 같아요
화자 2: 맞아요
화자 2: 바다가고싶어요!
화자 1: 동해바다 vs 서해바다
화자 2: 서해!
화자 1: 갯벌 좋아해요?
화자 2: 한번밖에 안가봤어요ㅠ
화자 2: 좋아하세요?
화자 1: 갯벌 진짜 꿀잼이에요 ㅋㅋㅋㅋ
화자 1: 거기서 게도 잡고 조개도 잡고~ 근데 옷 버릴 각오 해야되는 게 맴찢..
화자 2: 우와 가족끼리 같이 가고싶네요
화자 2: 버릴옷 필수,..
화자 1: 조카 있어요?
화자 1: 조카들이 많이 좋아할 것 같아요 ㅎㅎ
화자 2: 아직 조카가 없어요ㅜㅜ
화자 2: 아 애들이랑 같이가면 진짜 재밌겠네요
화자 1: 애들이랑 같이 가면 재밌는데ㅋㅋㅋ 자신을 포기해야해요
화자 1: 원주민 될 지도 몰라요 ㅎㅎ
'''
response1 = chain.invoke({"actual_dialouge": dialouge1})
print(response1)

### 3.2 nikluge-2025-대화 맥락 추론-train-000043

In [ ]:
dialouge2 = '''
화자 1: 화장품 오래된거 쓰면 안돼요 ㅋ
화자 2: 당연한거가지고
화자 2: 피부 안좋아진다
화자 1: 맞아요 그니까 집에있는 클렌징오일도 버려요 ㅋ
화자 2: 그거 오래 안되지 않았어?
화자 1: 작년에 샀던가... 근데
화자 1: 피부에 뭐 나는거같아요 바르니까
화자 2: 두드러기 나면 안좋은데
화자 1: 그니까 버리는게 좋겠어요
화자 1: 저 틴트도 새로 사야할거같은데
화자 2: 그거 너무 빨갛더라
화자 1: 사진 보셨죠 ㅠ 완전 안맞음
화자 2: 쥐잡아먹은줄
화자 2: 알았다
화자 1: 그절도예요?? ㅋㅋㅋㅋ 그정돈아니었던거같은데
화자 2: 심했어
화자 1: 엄마두 화장 빨갛게 하시면서
화자 2: 그정도는 아니야 엄마는 잘하지
화자 1: 좋은데 코디가 가끔 별로세요
화자 2: 같이 보러가서 골라줘야지
화자 1: 근데 제가 골라두 별로 ㅠ 맘에 안들어하시는거같은데...
화자 2: 그래도 마음에 들어
'''
response2 = chain.invoke({"actual_dialouge": dialouge2})
print(response2)

### 3.3 nikluge-2025-대화 맥락 추론-train-000095

In [ ]:
dialouge3 = '''
화자 2: 그러게
화자 1: 이번주 토요일 모임잇는데
화자 2: 수~금 비온다는데
화자 1: 엉 근데 토요일은 맑음
화자 2: 금요일에 다시 봐야한다
화자 1: 응 안그래도 금요일 날씨보고 토요일 모임할지 최종결정
화자 2: 잘했다
화자 1: 토요일에도 더워서 야외에서 만나도 안추울듯
화자 2: 야외에서 볼려고
화자 1: 응 좀 그늘지고 한곳에 있으면 더워도 참을수 잇을듯하다
화자 2: 그래 선크링도 잘 바르고
화자 1: 응 양산도 가져가서 쓰고 있을려고 ㅋㅋ
화자 2: 모자 쓰면 괜찮을텐데
화자 1: 모자도 쓰고 이중으로 자외선 차단
화자 2: 대단하네
화자 1: 갑자기 날씨가 너무 추워짐
'''
response3 = chain.invoke({"actual_dialouge": dialouge3})
print(response3)

### 3.4 nikluge-2025-대화 맥락 추론-train-000120

In [ ]:
dialouge4 = '''
화자 2: 오 정말? 무슨 일인데???
화자 1: 아 무슨 일인지 그동안 모랄ㅆ는데 이번에 일을줬어. 데이터 수집하고 웹사이트에 올리는거
화자 2: 빅데이터 관련 업무인가??낯선분야라 신기하다 일한지는 얼마나 됐어ㅓ??
화자 1: 빅데이터랑은 좀 다르더라 거긴 전문적이고 여긴 좀 덜 전문적 ㅋㅋ 이제 이주됐어
화자 2: 그렇구나 지금 한창 적응하고 잇겠네~ 일은 할 만해??
화자 1: 그동안 할만했는데 이제어려워졌었어..ㅠ
화자 2: 헉ㅠㅠ 앞날이 많이 힘들겠구나 머리가 아프겠는걸
화자 1: 그니까 어떠케 ㅠ 나 너무 바보같아서 아무것도 못하겟다능
화자 2: 아니야 처음하면 다들 똑같이 헤매지ㅠㅠ
화자 1: 마자..잘하기보다도 그냥..적당히만 하기라도 바랄뿐
화자 1: ㅠㅠ 난생 처음 보는 분야라너무 생소해..
화자 2: 너가 잘하길 응원할게
화자 2: 같이 일하는 동료들은 많아?
화자 1: 같이 5명 일해 나까지 5명  ㅎㅎㅎ...ㅎㅎㅎ...새롭다 새로워
화자 1: 너는 요새 일 괜찮아? 요새는ㅇ ㅓ때?
화자 2: 나는 늘 똑같지 뭐ㅎㅎ
'''
response4 = chain.invoke({"actual_dialouge": dialouge4})
print(response4)

전체 데이터셋은 